In [ ]:
%load_ext autoreload
%autoreload 2
import json
from src.agents.code_analyzer.code_analyzer_agent import CodeAnalyzerAgent
from src.agents.issue_creation.issue_creation_agent import IssueCreationAgent
from src.agents.new_implementer.implementer_agent import NewImplementerAgent
from src.agents.recording_bug_report.recording_bug_report_agent import (
    RecordingBugReportAgent,
)
from src.agents.utils.task.task import Task
from src.agents.utils.task.workflow import Workflow
from src.agents.validation.validation_agent import ValidationAgent
from src.schemas.core.common.recordings import RecordingCollection
from src.schemas.core.common.workspaces import Workspace, WorkspaceType
from src.agents.utils.task.debug.common import Debug
from src.agents.utils.task.debug.settings import DebugSettings
from src.agents.search.tree_agent import TreeAgent
from src.agents.search.brute_search_agent import BruteSearchAgent
from src.schemas.core.common.chat import ChatMessage
from src.schemas.account import User
from src.utils.debug.jupyter_utils import init_debug

init_debug()

user = User(id="6f11f0d5-3409-4a6c-8ace-f7f44725ec64", email="daniel@speck.sh")
spnix = "/Users/danielgeorge/Documents/work/sphinxbio/benchlytics"
wisdolia = "/Users/danielgeorge/Documents/work/wisdolia"
anki = "/Users/danielgeorge/Documents/work/ml/small-stuff/anki-question-gen"
plates_crud = "/Users/danielgeorge/Documents/work/random/plates-crud"
paige_site = (
    "/Users/danielgeorge/Documents/work/ml/small-stuff/speck/example-repos/paige"
)
expect_demo = (
    "/Users/danielgeorge/Documents/work/ml/small-stuff/speck/example-repos/eXpect-demo"
)
cal = "/Users/danielgeorge/Documents/work/ml/small-stuff/speck/example-repos/cal-demo"
cal_tsconfig_path = "/apps/web/tsconfig.json"
anki_frontend_dir = "/apps/web"
spnix_frontend_dir = "/frontend"
expect_frontend_dir = "/client"
paige_frontend_dir = "/apps/speck-site"
cal_frontend_dir = "/"
debug_settings = DebugSettings(
    user=user,
    created_repo_id="test-repo",
    package_manager="pnpm",
    port=4010,
    install_command="pnpm install",
    dev_command="pnpm dev",
    root_directory=cal_frontend_dir,
    workspace_path=cal,
    tsconfig_path=cal_tsconfig_path,
)
debug_config = Debug(
    original_repo_dir=cal,
    repo_dir=cal,
    env_file="tests/eval_repos/issue-tracker-eval-repo/.env",
    settings=debug_settings,
    initial_commit="a196aedfd18ad110c1918edff75ea8365c24c209",
    target_commit="0fdd5309a6d6ee3b1af9cf9c5bb085e4f4d8ed86",
)

# await session.initializer.initialize()
task = Task(
    user=user,
    workspace=Workspace(
        type=WorkspaceType.NEW,
        settings=debug_settings,
        name="debug-ai-frontend-engineer",
    ),
    git_repo_id=0,
    task_id="",
    issue_number=0,
    debug=debug_config,
)

import json
from pathlib import Path

# Dump results to JSON file
cal_replay_output = "src/agents/utils/task/interfaces/browsing/temp_script_output/cal/results_screenshot.json"
replay_output_path = Path(cal_replay_output)
replay_output_path.parent.mkdir(parents=True, exist_ok=True)

screenshot_expect = "src/agents/utils/task/interfaces/browsing/test_data/expect/recording_with_screenshot2_events.json"
selected_components_expect = "src/agents/utils/task/interfaces/browsing/test_data/expect/recording_with_selected_component.json"
selected_components_paige = "/Users/danielgeorge/Documents/work/ml/small-stuff/speck/paige/apps/api/src/agents/utils/task/interfaces/browsing/test_data/paige-site/results_selected_component.json"
cal_screenshot = (
    "src/agents/utils/task/interfaces/browsing/test_data/cal/results_screenshot.json"
)

with open(cal_screenshot, "r") as f:
    recording = json.load(f)
    data = RecordingCollection(recordings=[recording])

workflow = Workflow(
    user_message="",
    task=task,
    relevant_chats=[
        chat.model_dump()
        for chat in [ChatMessage.model_validate_json(chat) for chat in []]
    ],
    recordings=data,
    fully_autonomous=False,
    is_cloning_site=False,
    debug=debug_config,
)

task.current_workflow = workflow
task.tree_agent = TreeAgent(task)
await task.tree_agent.get_frontend_tree()
task.code_analyzer_agent = CodeAnalyzerAgent(task)
task.brute_search_agent = BruteSearchAgent(task)
task.new_implementer_agent = NewImplementerAgent(task)
task.issue_creation_agent = IssueCreationAgent(task)
task.validation_agent = ValidationAgent(task)
task.recording_bug_report_agent = RecordingBugReportAgent(task)
# await task.website.start()
recording = data.recordings[0]

In [13]:
from src.schemas.core.common.recordings import ReplayResponse

if replay_output_path.exists():
    with open(replay_output_path, "r") as f:
        results = ReplayResponse(**json.load(f))

# from src.agents.utils.task.interfaces.browsing.recorder_player import RecordingPlayer
# player = RecordingPlayer(task)
# results: ReplayResponse = await player.replay_recording(recording, delay_multiplier=1.0)

In [ ]:
from src.agents.recording_bug_report.utils.recording_artifact_extractor import (
    extract_replay_artifacts,
)
from src.schemas.core.common.recordings import RecordingArtifact, VideoSegmentArtifact


artifacts: list[RecordingArtifact] = await extract_replay_artifacts(results)
video_artifacts = [a for a in artifacts if isinstance(a, VideoSegmentArtifact)]

In [15]:
with open("temp_used_files.json", "r") as f:
    browser_files = json.load(f)
    cal_screenshot = "src/agents/utils/task/interfaces/browsing/test_data/cal/results_screenshot_used_files.json"
    # make the file or write to it
    with open(cal_screenshot, "w") as f:
        json.dump(browser_files, f)

In [ ]:
browser_files

In [ ]:
from src.agents.utils.gitignore_utils import filter_ignored_files

filtered_browser_files = await filter_ignored_files(task, browser_files)

In [ ]:
filtered_browser_files

In [ ]:
# from src.agents.recording_bug_report.recording_bug_report_agent import (
#     BugReportResult,
# )

task.recording_bug_report_agent.browser_files = filtered_browser_files
bug_report = await task.recording_bug_report_agent.generate_bug_report(
    recording, debug_file=replay_output_path
)

# print("Generated Bug Report:")
# print(bug_report)

# dump and reload
# import json

# with open("bug_report.json", "w") as f:
#     json.dump(bug_report.model_dump(), f)

# bug_report = BugReportResult(**json.load(open("bug_report.json")))

In [ ]:
issue = await task.issue_creation_agent.generate_issue_description(bug_report)

In [ ]:
from src.agents.validation.visual.visual_tester import VisualTester

# Run validation on the first recording
visual_tester = VisualTester(task, None)
validation_response = await visual_tester.validate_functionality()